In [ ]:
# ================================================================
#  ECG ARRHYTHMIA CLASSIFICATION — FULL PROJECT
#  -----------------------------------------------
#  PIPELINE:
#   1. Data Loading & Exploration (EDA)
#   2. Preprocessing & Feature Engineering
#   3. ML Models  (KNN, SVM, Random Forest, LightGBM)
#   4. ML Comparison Chart
#   5. 1D-CNN Deep Learning Model
#   6. Training Curves (Loss + Accuracy)
#   7. Confusion Matrix (ML best + CNN)
#   8. Classification Report
#   9. ROC Curves (per class)
#  10. Feature Importance (Random Forest)
#  11. Save all figures + models
#
#  CLASSES:  N=Normal  R=RBBB  V=PVC  L=LBBB
#
#  INSTALL:
#    pip install tensorflow lightgbm scikit-learn
#               matplotlib seaborn numpy pandas
#
#  RUN:
#    python ecg_full_project.py
# ================================================================

import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
warnings.filterwarnings('ignore')

# ── Reproducibility ─────────────────────────────────────────────
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED);  np.random.seed(SEED)

# ── Output folder ────────────────────────────────────────────────
OUT = "ecg_results"
os.makedirs(OUT, exist_ok=True)

print("=" * 60)
print("   ECG ARRHYTHMIA CLASSIFICATION — FULL PROJECT")
print("=" * 60)

# ================================================================
# CONFIGURATION  — edit only this block
# ================================================================
FOLDER_PATH = "/Users/ifty_sami/Downloads/shuju"  # ← your path
NUM_SAMPLES = 3500
EPOCHS      = 50
BATCH_SIZE  = 32
VAL_SPLIT   = 0.20      # 20% validation, 80% train

LABEL_MAP   = {"N": 0, "R": 1, "V": 2, "L": 3}
LABEL_NAMES = ["N (Normal)", "R (RBBB)", "V (PVC)", "L (LBBB)"]
SHORT       = ["N", "R", "V", "L"]
COLORS      = ["#4e79a7", "#f28e2b", "#e15759", "#76b7b2"]

# ================================================================
# STEP 1 — LOAD RAW DATA
# ================================================================
print("\n[1/10] Loading data ...")

all_files = [f for f in os.listdir(FOLDER_PATH) if f.endswith(".npy")]
random.shuffle(all_files)
all_files = all_files[:NUM_SAMPLES]

signals, labels = [], []
for f in all_files:
    parts = f.split("_")
    if len(parts) < 2 or parts[1] not in LABEL_MAP:
        continue
    data = np.load(os.path.join(FOLDER_PATH, f))
    signals.append(data[:, 0].astype(np.float32))
    labels.append(LABEL_MAP[parts[1]])

X_raw = np.array(signals, dtype=np.float32)   # (N, 284)
y     = np.array(labels,  dtype=np.int32)

print(f"    Loaded  {X_raw.shape[0]} samples  |  length = {X_raw.shape[1]}")
for k, v in LABEL_MAP.items():
    print(f"    {k}: {(y==v).sum()} samples")

# ================================================================
# STEP 2 — EDA: SAMPLE SIGNAL PLOT
# ================================================================
print("\n[2/10] Exploratory Data Analysis ...")

fig, axes = plt.subplots(2, 2, figsize=(12, 6))
fig.suptitle("Figure 1: Sample ECG Signals per Class",
             fontsize=14, fontweight='bold')
for ax, (name, idx), color in zip(axes.flat, LABEL_MAP.items(), COLORS):
    sample = X_raw[y == idx][0]
    ax.plot(sample, color=color, linewidth=1.2)
    ax.set_title(f"Class {name}", fontsize=12, fontweight='bold')
    ax.set_xlabel("Sample Point", fontsize=10)
    ax.set_ylabel("Amplitude",    fontsize=10)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Class distribution bar chart
fig, ax = plt.subplots(figsize=(6, 4))
counts = [(y == v).sum() for v in LABEL_MAP.values()]
bars = ax.bar(LABEL_NAMES, counts, color=COLORS, edgecolor='black', linewidth=0.7)
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(cnt), ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_xlabel("Class",  fontsize=12)
ax.set_ylabel("Count",  fontsize=12)
ax.set_title("Figure 2: Class Distribution", fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()
print("    EDA figures saved.")

# ================================================================
# STEP 3 — PREPROCESSING & FEATURE ENGINEERING
# ================================================================
print("\n[3/10] Preprocessing & feature engineering ...")

from sklearn.preprocessing import StandardScaler

# ── (A) Normalise raw signals ────────────────────────────────────
X_norm = (X_raw - X_raw.mean(axis=1, keepdims=True)) / \
         (X_raw.std(axis=1,  keepdims=True) + 1e-8)

# ── (B) Hand-crafted statistical features for ML models ─────────
def extract_features(signals):
    feats = []
    for s in signals:
        f = [
            np.mean(s),
            np.std(s),
            np.min(s),
            np.max(s),
            np.max(s) - np.min(s),          # peak-to-peak
            np.percentile(s, 25),
            np.percentile(s, 75),
            np.percentile(s, 75) - np.percentile(s, 25),  # IQR
            np.mean(np.abs(s)),              # MAV
            np.sqrt(np.mean(s**2)),          # RMS
            np.mean(np.diff(s)),             # mean 1st derivative
            np.std(np.diff(s)),              # std  1st derivative
            np.sum(s**2),                    # energy
            float(np.sum(np.diff(np.sign(s)) != 0)),  # zero crossings
        ]
        feats.append(f)
    return np.array(feats, dtype=np.float32)

X_feat = extract_features(X_norm)
print(f"    Feature matrix: {X_feat.shape}")

scaler = StandardScaler()
X_feat_scaled = scaler.fit_transform(X_feat)

# ── Train/Test split ─────────────────────────────────────────────
from sklearn.model_selection import train_test_split

(X_tr_feat, X_te_feat,
 X_tr_raw,  X_te_raw,
 y_tr,      y_te) = train_test_split(
    X_feat_scaled, X_norm, y,
    test_size=0.30, random_state=SEED, stratify=y
)
print(f"    Train={len(y_tr)}   Test={len(y_te)}")

# ================================================================
# STEP 4 — ML MODELS
# ================================================================
print("\n[4/10] Training ML models ...")

from sklearn.neighbors         import KNeighborsClassifier
from sklearn.svm               import SVC
from sklearn.ensemble          import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics           import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics           import roc_curve, auc
from sklearn.preprocessing     import label_binarize
import time

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("    LightGBM not found — skipping (install with: pip install lightgbm)")

ml_models = {
    "KNN (k=5)":        KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    "SVM (RBF)":        SVC(kernel='rbf', C=10, gamma='scale',
                            probability=True, random_state=SEED),
    "Random Forest":    RandomForestClassifier(n_estimators=200,
                            random_state=SEED, n_jobs=-1),
    "Gradient Boost":   GradientBoostingClassifier(n_estimators=200,
                            random_state=SEED),
}
if HAS_LGB:
    ml_models["LightGBM"] = lgb.LGBMClassifier(
        n_estimators=200, random_state=SEED, n_jobs=-1, verbose=-1)

ml_results = {}
best_ml_acc = 0
best_ml_name = ""
best_ml_model = None

for name, clf in ml_models.items():
    t0 = time.time()
    clf.fit(X_tr_feat, y_tr)
    pred = clf.predict(X_te_feat)
    acc  = accuracy_score(y_te, pred)
    sec  = time.time() - t0
    ml_results[name] = {"acc": acc, "pred": pred, "time": sec, "model": clf}
    if acc > best_ml_acc:
        best_ml_acc, best_ml_name, best_ml_model = acc, name, clf
    print(f"    {name:20s}  acc={acc:.4f}  ({sec:.1f}s)")

print(f"\n    Best ML model: {best_ml_name}  ({best_ml_acc:.4f})")

# ================================================================
# STEP 5 — ML COMPARISON CHART
# ================================================================
print("\n[5/10] Plotting ML comparison ...")

names = list(ml_results.keys())
accs  = [ml_results[n]["acc"] * 100 for n in names]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(names, accs, color=plt.cm.Blues(
    np.linspace(0.4, 0.85, len(names))),
    edgecolor='black', linewidth=0.6, height=0.55)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f"{acc:.2f}%", va='center', fontsize=11, fontweight='bold')
ax.set_xlabel("Accuracy (%)", fontsize=12)
ax.set_title("Figure 3: ML Model Comparison", fontsize=13, fontweight='bold')
ax.set_xlim(0, 105)
ax.grid(axis='x', alpha=0.4)
plt.tight_layout()
plt.show()
plt.close()
# ================================================================
# STEP 6 — BEST ML CONFUSION MATRIX
# ================================================================
print("\n[6/10] Confusion matrices ...")

def plot_cm(y_true, y_pred, title, filename):
    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm_norm, annot=False, cmap='Blues',
                linewidths=0.5, linecolor='gray',
                xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
                ax=ax, vmin=0, vmax=100,
                cbar_kws={'label': 'Percentage (%)'})
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            col = 'white' if cm_norm[i, j] > 55 else 'black'
            ax.text(j+0.5, i+0.5,
                    f"{cm[i,j]}\n({cm_norm[i,j]:.1f}%)",
                    ha='center', va='center',
                    fontsize=11, color=col, fontweight='bold')
    ax.set_xlabel('Predicted Label', fontsize=13, labelpad=10)
    ax.set_ylabel('True Label',      fontsize=13, labelpad=10)
    ax.set_title(title, fontsize=14, fontweight='bold', pad=14)
    plt.xticks(rotation=30, ha='right', fontsize=10)
    plt.yticks(rotation=0,  fontsize=10)
    plt.tight_layout()
    plt.show()  
    plt.close() 

best_pred = ml_results[best_ml_name]["pred"]
plot_cm(y_te, best_pred,
        f"Figure 4: Confusion Matrix — {best_ml_name}",
        "fig4_ml_confusion_matrix.png")
print(f"    ML confusion matrix saved ({best_ml_name})")

# ================================================================
# STEP 7 — FEATURE IMPORTANCE (Random Forest)
# ================================================================
print("\n[7/10] Feature importance ...")

feat_names = [
    "Mean","Std","Min","Max","Peak-to-Peak",
    "Q25","Q75","IQR","MAV","RMS",
    "Mean dX","Std dX","Energy","Zero Crossings"
]
rf_model = ml_results["Random Forest"]["model"]
importances = rf_model.feature_importances_
idx = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(range(len(feat_names)),
       importances[idx],
       color=plt.cm.Greens(np.linspace(0.4, 0.85, len(feat_names))),
       edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(feat_names)))
ax.set_xticklabels([feat_names[i] for i in idx],
                    rotation=40, ha='right', fontsize=9)
ax.set_ylabel("Importance", fontsize=12)
ax.set_title("Figure 5: Feature Importance (Random Forest)",
             fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()
plt.close()
print("    Feature importance plot saved (Random Forest)")  

# ================================================================
# STEP 8 — ROC CURVES (best ML model)
# ================================================================
print("\n[8/10] ROC curves ...")

y_te_bin = label_binarize(y_te, classes=[0, 1, 2, 3])
if hasattr(best_ml_model, "predict_proba"):
    y_score = best_ml_model.predict_proba(X_te_feat)
else:
    y_score = best_ml_model.decision_function(X_te_feat)

fig, ax = plt.subplots(figsize=(7, 5))
for i, (name, color) in enumerate(zip(LABEL_NAMES, COLORS)):
    fpr, tpr, _ = roc_curve(y_te_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f"{name}  (AUC = {roc_auc:.3f})")
ax.plot([0,1],[0,1], 'k--', linewidth=1)
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate",  fontsize=12)
ax.set_title(f"Figure 6: ROC Curves — {best_ml_name}",
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()
plt.close()
print(f"    ROC curves saved ({best_ml_name})") 

# ================================================================
# STEP 9 — 1D-CNN DEEP LEARNING
# ================================================================
print("\n[9/10] Training 1D-CNN ...")

import tensorflow as tf
tf.random.set_seed(SEED)
from tensorflow import keras
from tensorflow.keras import layers

# ── Prepare CNN inputs ───────────────────────────────────────────
X_cnn = X_norm[:, :, np.newaxis].astype(np.float32)  # (N, 284, 1)
y_cat = keras.utils.to_categorical(y, len(LABEL_MAP))

(X_cnn_tr, X_cnn_val,
 y_cnn_tr,  y_cnn_val,
 y_tr_idx,  y_val_idx) = train_test_split(
    X_cnn, y_cat, y,
    test_size=VAL_SPLIT, random_state=SEED, stratify=y
)

# ── Build model ──────────────────────────────────────────────────
def build_cnn(input_shape, n_cls):
    inp = keras.Input(shape=input_shape, name="ecg_input")
    x = layers.Conv1D(32,  5, padding='same', activation='relu', name='conv1')(inp)
    x = layers.BatchNormalization(name='bn1')(x)
    x = layers.MaxPooling1D(2, name='pool1')(x)
    x = layers.Dropout(0.25, name='drop1')(x)

    x = layers.Conv1D(64,  5, padding='same', activation='relu', name='conv2')(x)
    x = layers.BatchNormalization(name='bn2')(x)
    x = layers.MaxPooling1D(2, name='pool2')(x)
    x = layers.Dropout(0.25, name='drop2')(x)

    x = layers.Conv1D(128, 3, padding='same', activation='relu', name='conv3')(x)
    x = layers.BatchNormalization(name='bn3')(x)
    x = layers.MaxPooling1D(2, name='pool3')(x)
    x = layers.Dropout(0.25, name='drop3')(x)

    x = layers.Conv1D(256, 3, padding='same', activation='relu', name='conv4')(x)
    x = layers.BatchNormalization(name='bn4')(x)
    x = layers.GlobalAveragePooling1D(name='gap')(x)

    x = layers.Dense(128, activation='relu', name='fc1')(x)
    x = layers.Dropout(0.40, name='drop_fc')(x)
    out = layers.Dense(n_cls, activation='softmax', name='output')(x)
    return keras.Model(inputs=inp, outputs=out, name="ECG_1D_CNN")

model = build_cnn((X_cnn_tr.shape[1], 1), len(LABEL_MAP))
model.summary()

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5,
        verbose=1, min_lr=1e-6),
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=12,
        restore_best_weights=True, verbose=1),
    keras.callbacks.ModelCheckpoint(
        f'{OUT}/ecg_cnn_best.h5',
        monitor='val_accuracy', save_best_only=True, verbose=1),
]

history = model.fit(
    X_cnn_tr, y_cnn_tr,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_cnn_val, y_cnn_val),
    callbacks=callbacks,
    verbose=1
)

ep = range(1, len(history.history['loss']) + 1)

# ── Loss curve ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(ep, history.history['loss'],     color='red',  linewidth=2,
        label='Training Loss')
ax.plot(ep, history.history['val_loss'], color='blue', linewidth=2,
        linestyle='--', label='Validation Loss')
ax.set_xlabel('Epoch', fontsize=13)
ax.set_ylabel('Loss',  fontsize=13)
ax.set_title('Figure 7: Loss Rate during Training Process',
             fontsize=13, fontweight='bold')
ax.set_xlim(0, len(ep)+1);  ax.set_ylim(0, None)
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, linestyle='-', alpha=0.4)
plt.tight_layout()
plt.show()

# ── Accuracy curve ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(ep, history.history['accuracy'],     color='blue',   linewidth=2,
        label='Training Accuracy')
ax.plot(ep, history.history['val_accuracy'], color='orange', linewidth=2,
        linestyle='--', label='Validation Accuracy')
ax.set_xlabel('Epoch',    fontsize=13)
ax.set_ylabel('Accuracy', fontsize=13)
ax.set_title('Figure 8: Accuracy Rate during Training Process',
             fontsize=13, fontweight='bold')
ax.set_xlim(0, len(ep)+1);  ax.set_ylim(0.40, 1.02)
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, linestyle='-', alpha=0.4)
plt.tight_layout()
plt.show()


# ── CNN Confusion matrix ─────────────────────────────────────────
y_cnn_pred = np.argmax(model.predict(X_cnn_val, verbose=0), axis=1)
plot_cm(y_val_idx, y_cnn_pred,
        "Figure 9: Confusion Matrix — 1D-CNN",
        "fig9_cnn_confusion_matrix.png")

# ── CNN ROC curves ───────────────────────────────────────────────
y_val_bin  = label_binarize(y_val_idx, classes=[0,1,2,3])
y_cnn_prob = model.predict(X_cnn_val, verbose=0)

fig, ax = plt.subplots(figsize=(7, 5))
for i, (name, color) in enumerate(zip(LABEL_NAMES, COLORS)):
    fpr, tpr, _ = roc_curve(y_val_bin[:, i], y_cnn_prob[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f"{name}  (AUC = {roc_auc:.3f})")
ax.plot([0,1],[0,1], 'k--', linewidth=1)
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate",  fontsize=12)
ax.set_title("Figure 10: ROC Curves — 1D-CNN",
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()


# ================================================================
# STEP 10 — FINAL SUMMARY TABLE
# ================================================================
print("\n[10/10] Final comparison summary ...")

cnn_loss, cnn_acc = model.evaluate(X_cnn_val, y_cnn_val, verbose=0)

summary_rows = []
for name in ml_results:
    summary_rows.append({
        "Model": name,
        "Type":  "Machine Learning",
        "Accuracy (%)": round(ml_results[name]["acc"] * 100, 2),
        "Time (s)": round(ml_results[name]["time"], 1),
    })
summary_rows.append({
    "Model": "1D-CNN",
    "Type":  "Deep Learning",
    "Accuracy (%)": round(cnn_acc * 100, 2),
    "Time (s)": "—",
})

df_summary = pd.DataFrame(summary_rows)
df_summary = df_summary.sort_values("Accuracy (%)", ascending=False)
print("\n" + df_summary.to_string(index=False))
df_summary.to_csv(f"{OUT}/summary_table.csv", index=False)

# Summary bar chart
fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = ["#4e79a7"] * (len(summary_rows)-1) + ["#e15759"]
bars = ax.bar(df_summary["Model"],
              df_summary["Accuracy (%)"],
              color=bar_colors, edgecolor='black', linewidth=0.7, width=0.55)
for bar, acc in zip(bars, df_summary["Accuracy (%)"]):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.2,
            f"{acc:.2f}%", ha='center', va='bottom',
            fontsize=11, fontweight='bold')
ax.set_ylabel("Accuracy (%)", fontsize=13)
ax.set_title("Figure 11: All Models Comparison (ML vs CNN)",
             fontsize=13, fontweight='bold')
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.4)
ax.legend(handles=[
    plt.Rectangle((0,0),1,1, color="#4e79a7", label="Machine Learning"),
    plt.Rectangle((0,0),1,1, color="#e15759", label="Deep Learning (CNN)"),
], fontsize=11)
plt.xticks(rotation=15, ha='right', fontsize=10)
plt.tight_layout()
plt.show()


# ── Classification reports ───────────────────────────────────────
print("\n" + "="*60)
print(f"  Classification Report — {best_ml_name}")
print("="*60)
print(classification_report(y_te, best_pred, target_names=SHORT, digits=4))

print("\n" + "="*60)
print("  Classification Report — 1D-CNN")
print("="*60)
print(classification_report(y_val_idx, y_cnn_pred,
                             target_names=SHORT, digits=4))

# ── Save models ──────────────────────────────────────────────────
model.save(f"{OUT}/ecg_cnn_model.h5")
import joblib
joblib.dump(best_ml_model, f"{OUT}/ecg_{best_ml_name.replace(' ','_').lower()}_model.pkl")
joblib.dump(scaler,        f"{OUT}/scaler.pkl")

# ── Final summary print ──────────────────────────────────────────
print("\n" + "="*60)
print("  ALL FILES SAVED TO:", OUT)
print("="*60)
files = sorted(os.listdir(OUT))
for f in files:
    size = os.path.getsize(os.path.join(OUT, f))
    print(f"    {f:45s} {size/1024:8.1f} KB")

print("\n✅  FULL PROJECT COMPLETE!\n")


   ECG ARRHYTHMIA CLASSIFICATION — FULL PROJECT

[1/10] Loading data ...
    Loaded  2817 samples  |  length = 284
    N: 715 samples
    R: 720 samples
    V: 704 samples
    L: 678 samples

[2/10] Exploratory Data Analysis ...
    EDA figures saved.

[3/10] Preprocessing & feature engineering ...
    Feature matrix: (2817, 14)
    Train=1971   Test=846

[4/10] Training ML models ...
    LightGBM not found — skipping (install with: pip install lightgbm)
    KNN (k=5)             acc=0.8995  (0.0s)
    SVM (RBF)             acc=0.9433  (0.2s)
    Random Forest         acc=0.9515  (0.2s)
    Gradient Boost        acc=0.9563  (4.1s)

    Best ML model: Gradient Boost  (0.9563)

[5/10] Plotting ML comparison ...

[6/10] Confusion matrices ...
    ML confusion matrix saved (Gradient Boost)

[7/10] Feature importance ...

[8/10] ROC curves ...

[9/10] Training 1D-CNN ...


Model: "ECG_1D_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ ecg_input (InputLayer)          │ (None, 284, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv1D)                  │ (None, 284, 32)        │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1 (BatchNormalization)        │ (None, 284, 32)        │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling1D)            │ (None, 142, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop1 (Dropout)                 │ (None, 142, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv1D)                  │ (None, 142, 64)        │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2 (BatchNormalization)        │ (None, 142, 64)        │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling1D)            │ (None, 71, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop2 (Dropout)                 │ (None, 71, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3 (Conv1D)                  │ (None, 71, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn3 (BatchNormalization)        │ (None, 71, 128)        │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool3 (MaxPooling1D)            │ (None, 35, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop3 (Dropout)                 │ (None, 35, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv4 (Conv1D)                  │ (None, 35, 256)        │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn4 (BatchNormalization)        │ (None, 35, 256)        │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap (GlobalAveragePooling1D)    │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop_fc (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 169,092 (660.52 KB)

 Trainable params: 168,132 (656.77 KB)

...

71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9831 - loss: 0.0498 - val_accuracy: 0.9876 - val_loss: 0.0268 - learning_rate: 0.0010
Epoch 13/50
68/71 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9839 - loss: 0.0475
Epoch 13: val_accuracy did not improve from 0.98759
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9862 - loss: 0.0434 - val_accuracy: 0.9840 - val_loss: 0.0579 - learning_rate: 0.0010
Epoch 14/50
69/71 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9927 - loss: 0.0338
Epoch 14: val_accuracy did not improve from 0.98759
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9889 - loss: 0.0395 - val_accuracy: 0.9787 - val_loss: 0.0698 - learning_rate: 0.0010
Epoch 15/50
70/71 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9817 - loss: 0.0484
Epoch 15: val_accuracy did not improve from 0.98759
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9840 - loss: 0.0474 - val_accuracy: 0.9840 - val_loss: 0.0387 - learning_rate: 0.0010
Epoch 16/50
70/71 ━━━━━━━━

71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9942 - loss: 0.0193 - val_accuracy: 0.9947 - val_loss: 0.0130 - learning_rate: 5.0000e-04
Epoch 19/50
70/71 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9948 - loss: 0.0185
Epoch 19: val_accuracy did not improve from 0.99468
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9947 - loss: 0.0188 - val_accuracy: 0.9911 - val_loss: 0.0205 - learning_rate: 5.0000e-04
Epoch 20/50
68/71 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9964 - loss: 0.0162
Epoch 20: val_accuracy did not improve from 0.99468
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9960 - loss: 0.0138 - val_accuracy: 0.9929 - val_loss: 0.0175 - learning_rate: 5.0000e-04
Epoch 21/50
70/71 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9949 - loss: 0.0191
Epoch 21: val_accuracy improved from 0.99468 to 0.99645, saving model to ecg_results/ecg_cnn_best.h5


71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9960 - loss: 0.0164 - val_accuracy: 0.9965 - val_loss: 0.0123 - learning_rate: 5.0000e-04
Epoch 22/50
70/71 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9946 - loss: 0.0175
Epoch 22: val_accuracy did not improve from 0.99645
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9960 - loss: 0.0185 - val_accuracy: 0.9805 - val_loss: 0.0509 - learning_rate: 5.0000e-04
Epoch 23/50
68/71 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9961 - loss: 0.0177
Epoch 23: val_accuracy did not improve from 0.99645
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9951 - loss: 0.0175 - val_accuracy: 0.9894 - val_loss: 0.0241 - learning_rate: 5.0000e-04
Epoch 24/50
70/71 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9971 - loss: 0.0115
Epoch 24: val_accuracy did not improve from 0.99645
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9973 - loss: 0.0105 - val_accuracy: 0.9947 - val_loss: 0.0187 - learning_rate: 5.0000e-04
Epoch 25/5

71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9938 - loss: 0.0202 - val_accuracy: 0.9982 - val_loss: 0.0161 - learning_rate: 5.0000e-04
Epoch 27/50
70/71 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9965 - loss: 0.0128
Epoch 27: val_accuracy did not improve from 0.99823
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9947 - loss: 0.0161 - val_accuracy: 0.9947 - val_loss: 0.0169 - learning_rate: 2.5000e-04
Epoch 28/50
70/71 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9957 - loss: 0.0138
Epoch 28: val_accuracy did not improve from 0.99823
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9964 - loss: 0.0113 - val_accuracy: 0.9965 - val_loss: 0.0157 - learning_rate: 2.5000e-04
Epoch 29/50
69/71 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9956 - loss: 0.0093
Epoch 29: val_accuracy did not improve from 0.99823
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9973 - loss: 0.0073 - val_accuracy: 0.9947 - val_loss: 0.0163 - learning_rate: 2.5000e-04
Epoch 30/5


[10/10] Final comparison summary ...

         Model             Type  Accuracy (%) Time (s)
        1D-CNN    Deep Learning         99.65        —
Gradient Boost Machine Learning         95.63      4.1
 Random Forest Machine Learning         95.15      0.2
     SVM (RBF) Machine Learning         94.33      0.2
     KNN (k=5) Machine Learning         89.95      0.0

  Classification Report — Gradient Boost
              precision    recall  f1-score   support

           N     1.0000    0.9814    0.9906       215
           R     0.9581    0.9537    0.9559       216
           V     0.9565    0.9384    0.9474       211
           L     0.9108    0.9510    0.9305       204

    accuracy                         0.9563       846
   macro avg     0.9564    0.9561    0.9561       846
weighted avg     0.9570    0.9563    0.9565       846


  Classification Report — 1D-CNN
              precision    recall  f1-score   support

           N     1.0000    0.9930    0.9965       143
           